##### The Primary Thesis:

I want to use my ML work thus far but this time, instead of informing equity trading decisions, the models will govern a
volatility trading strategy.

A few things to test before developing the model:

- What does the volatility landscape look like, where is the strategy's edge? (Compare the shapes of implied vol vs realised vol)


In [32]:
import numpy as np
import pandas as pd
import os
import plotly.graph_objects as go
from datetime import date

spy_csv_dir = "~/python-projects/ed-backtest/backtester/data/sp_constituents/SPY.csv"
vix_csv_dir = "~/python-projects/ed-backtest/backtester/data/sp_constituents/^VIX.csv"

spy_df = pd.read_csv(os.path.expanduser(spy_csv_dir), parse_dates=['Date'])
vix_df = pd.read_csv(os.path.expanduser(vix_csv_dir), parse_dates=['Date'])

spy_df['Date'] = pd.to_datetime(spy_df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
vix_df['Date'] = pd.to_datetime(vix_df['Date'], utc=True).dt.tz_localize(None).dt.normalize()

spy_df['Log_Returns'] = np.log(spy_df['Close'] / spy_df['Close'].shift(1))

window = 21
spy_df['RV'] = spy_df['Log_Returns'].rolling(window).std() * np.sqrt(252) * 100 #IV reflects 30 day options, so rolling 30d is used
merge_df = pd.merge(spy_df[['Date', 'RV']], vix_df[['Date', 'Close']], on='Date', how='inner', suffixes=('', '_VIX'))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=merge_df['Date'], y=merge_df['RV'], name=f"RV: {window}d"
))
fig.add_trace(go.Scatter(
    x=merge_df['Date'], y=merge_df['Close'], name="IV"
))
fig.show()
